# CrewAI Patterns — Agents, Tasks, Crews, and Process Types

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/23-crewai-notebook/crewai_workbook.ipynb)

---

## What is CrewAI?

CrewAI is a multi-agent framework built on the **role-based organization** metaphor.
Where LangGraph asks you to define a state machine (nodes + edges), CrewAI asks you to
describe a **team**: who are the agents, what are their roles, what tasks need doing,
and how should the team work together?

```
LangGraph mental model          CrewAI mental model
─────────────────────           ─────────────────────
State machine                   Organization chart
Nodes = processing steps        Agents = team members with roles
Edges = control flow            Tasks = assignments with expected outputs
You wire the graph              Crew orchestrates automatically
```

### What you will learn

- How to define `Agent`s with role, goal, backstory, and tools
- How to define `Task`s with descriptions and expected outputs
- How `Crew` orchestrates agents and tasks
- The difference between `Process.sequential` and `Process.hierarchical`
- How to write custom tools and attach them to agents


In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "crewai>=0.76.0",
        "langchain-openai==0.3.33",
        "python-dotenv==1.1.1",
    ], check=True)
    print("Dependencies installed.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# Colab: add OPENAI_API_KEY to Secrets (left sidebar key icon)
# Local: create a .env file with OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Key loaded from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Key loaded from .env file.")

---

## Part 1 — Agents: The Building Blocks

---

An **Agent** in CrewAI is defined by four things:

| Field | What it does | Why it matters |
|-------|-------------|----------------|
| `role` | The job title | Focuses the LLM on a domain |
| `goal` | What the agent is trying to achieve | Sets the objective for every action |
| `backstory` | Background and expertise | Shapes the LLM's persona and reasoning style |
| `llm` | The language model to use | Lets you use different models per agent |

The **backstory** is the most underrated field. A generic backstory produces generic output;
a specific backstory with domain expertise produces work that matches that role.

In [ ]:
from crewai import Agent, LLM

llm = LLM(model="openai/gpt-5-nano")

# A researcher with a specific domain and expertise
researcher = Agent(
    role="Technology Trend Researcher",
    goal="Find and summarize key developments in AI and software engineering",
    backstory=(
        "You are a senior technology analyst with 10 years of experience tracking "
        "AI and software trends. You synthesize complex technical topics into clear, "
        "accurate summaries backed by specific examples."
    ),
    llm=llm,
    verbose=False,
)

# A writer with a contrasting style
writer = Agent(
    role="Technical Content Writer",
    goal="Write engaging, accurate technical content for developer audiences",
    backstory=(
        "You are a developer advocate who writes for engineers. You avoid jargon, "
        "prefer concrete examples over abstractions, and always include a practical "
        "takeaway in every piece you write."
    ),
    llm=llm,
    verbose=False,
)

print(f"Researcher role: {researcher.role}")
print(f"Writer role:     {writer.role}")

---

## Part 2 — Tasks and the Sequential Process

---

A **Task** defines a unit of work: what should be done, what the output should look like,
and which agent should do it. Tasks in a sequential crew run in the order they are listed,
and each task can access the output of the previous one via `context`.

```
SEQUENTIAL PROCESS

Task 1: Research         Task 2: Write
  agent: researcher  ──►  agent: writer
  output: bullet list     context: [task_1]
                          output: article draft
```

In [ ]:
from crewai import Task, Crew, Process

TOPIC = "the rise of LLM-based coding agents in 2024-2025"

research_task = Task(
    description=f"Research {TOPIC}. Identify 3-5 key trends, notable tools, and their impact.",
    expected_output="A bullet-point list of 3-5 key trends with one concrete example each.",
    agent=researcher,
)

write_task = Task(
    description=(
        f"Using the research provided, write a concise 150-word blog intro about {TOPIC}. "
        "Open with a hook, cover 2 key trends, end with a call to action."
    ),
    expected_output="A 150-word blog introduction paragraph.",
    agent=writer,
    context=[research_task],  # writer receives researcher's output as context
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()
print(result.raw)

---

## Part 3 — The Hierarchical Process

---

In a **hierarchical process**, CrewAI creates a **manager agent** automatically.
The manager reads the task list and decides which worker agent to assign each task to,
and in what order. You don't wire the routing — the manager LLM does.

```
HIERARCHICAL PROCESS

            Manager Agent (auto-created)
                    |
       +────────────+────────────+
       |                        |
  researcher               writer
  (assigned tasks          (assigned tasks
   by manager)              by manager)
```

### When to use which process

| Process | Best for | Tradeoff |
|---------|----------|----------|
| `sequential` | Defined pipelines with clear handoffs | Rigid ordering; no dynamic routing |
| `hierarchical` | Open-ended tasks where agent assignment is non-obvious | Extra LLM call for the manager; less predictable |

**Contrast to LangGraph:** `sequential` maps to a linear graph; `hierarchical` maps to
the supervisor/worker pattern from example 19 — but CrewAI handles the routing logic
automatically, while LangGraph requires you to write the routing function explicitly.

In [ ]:
# Hierarchical crew: same agents and tasks, different process.
# The manager LLM decides which agent gets each task.

hierarchical_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.hierarchical,
    manager_llm=llm,   # the LLM that plays the manager role
    verbose=False,
)

result_h = hierarchical_crew.kickoff()
print(result_h.raw)

---

## Part 4 — Custom Tools

---

Agents become far more useful when they have **tools** — callable functions that let them
interact with the world beyond the LLM's training data.

CrewAI tools use the same `@tool` decorator as LangChain. Any LangChain tool also works.
The agent decides when to call a tool based on its goal and the current task description.

In [ ]:
from crewai.tools import tool

# A simple custom tool — a local knowledge base lookup.
KNOWLEDGE_BASE = {
    "langgraph": "LangGraph is a framework for building stateful multi-agent applications using graph-based workflows.",
    "crewai": "CrewAI is a role-based multi-agent framework focused on team coordination and task delegation.",
    "autogen": "AutoGen is a Microsoft framework for multi-agent conversation using ConversableAgent.",
}

@tool("lookup_framework")
def lookup_framework(framework_name: str) -> str:
    """Look up a description of an AI agent framework by name."""
    key = framework_name.lower().strip()
    return KNOWLEDGE_BASE.get(key, f"No entry found for '{framework_name}'.")

# Attach the tool to a new agent
framework_expert = Agent(
    role="AI Framework Specialist",
    goal="Accurately compare and explain AI agent frameworks",
    backstory="You are an expert in agentic AI frameworks with deep knowledge of LangGraph, CrewAI, and AutoGen.",
    llm=llm,
    tools=[lookup_framework],
    verbose=False,
)

compare_task = Task(
    description="Look up LangGraph and CrewAI and write a one-paragraph comparison of their core design philosophies.",
    expected_output="A concise 80-word comparison paragraph.",
    agent=framework_expert,
)

solo_crew = Crew(
    agents=[framework_expert],
    tasks=[compare_task],
    process=Process.sequential,
    verbose=False,
)

result_tool = solo_crew.kickoff()
print(result_tool.raw)

---

## Exercises

### Exercise 1 — Add a fact-checker
Add a third agent (`fact_checker`) to the research/writing crew that reviews the writer's
draft and flags any factual claims that need a source. Add a third task with
`context=[write_task]`.

### Exercise 2 — Convert to hierarchical
Take the three-agent crew from Exercise 1 and switch it to `Process.hierarchical`.
Observe whether the manager assigns tasks in the same order as the sequential version.

### Exercise 3 — Custom search tool
Replace the `KNOWLEDGE_BASE` dict with a DuckDuckGo search tool
(`from langchain_community.tools import DuckDuckGoSearchRun`) and attach it to the
researcher. Run the research task again and compare the quality of the output.

---

## What's Next?

- **Example 15 — CrewAI Research Crew** ([`../15-crewai-research-crew/`](../15-crewai-research-crew/)): a complete end-to-end research pipeline using CrewAI with web search tools
- **Example 19 — Multi-Agent Notebook** ([`../19-multi-agent-notebook/`](../19-multi-agent-notebook/)): the LangGraph supervisor/worker pattern — compare the explicit graph wiring to CrewAI's declarative approach
- **Example 21 — AutoGen Debate** ([`../21-autogen-debate/`](../21-autogen-debate/)): a third framework model — direct agent-to-agent conversation without a graph or crew
